In [1]:
from google.colab import files
uploaded = files.upload()

Saving Unique Words Data.xlsx to Unique Words Data.xlsx


In [3]:
import pandas as pd

df = pd.read_excel("Unique Words Data.xlsx")  # replace name
df.head()

,word
0,है
1,तो
2,में
3,जी
4,हैं


In [4]:
print(df.columns)

Index(['word'], dtype='object')


In [5]:
words_list = df.iloc[:,0].astype(str).tolist()
print("Total words:", len(words_list))

Total words: 177509


In [ ]:
import re

english_like = ["कंप्यूटर", "इंटरव्यू", "जॉब", "फिल्म", "प्रॉब्लम", "सॉल्व"]

def classify_word(word):
    # non-Devanagari
    if not re.match(r'^[\u0900-\u097F]+$', word):
        return "incorrect", "high", "non-Devanagari characters"

    # repeated characters (noise)
    if re.search(r'(.)\1{2,}', word):
        return "incorrect", "high", "character repetition"

    # too short or too long
    if len(word) < 2 or len(word) > 20:
        return "incorrect", "medium", "length issue"

    # valid English words in Devanagari
    if word in english_like:
        return "correct", "high", "English word in Devanagari"

    # default case
    return "correct", "medium", "looks valid"

In [7]:
results = []

for w in words_list:
    label, confidence, reason = classify_word(w)
    results.append((w, label, confidence, reason))

df_words = pd.DataFrame(results, columns=["word", "label", "confidence", "reason"])

df_words.head()

,word,label,confidence,reason
0,है,correct,medium,looks valid
1,तो,correct,medium,looks valid
2,में,correct,medium,looks valid
3,जी,correct,medium,looks valid
4,हैं,correct,medium,looks valid


In [9]:
correct_count = len(df_words[df_words["label"] == "correct"])
print("Total correct words:", correct_count)

Total correct words: 154510


In [10]:
def classify_word(word):
    import re

    # non-Devanagari
    if not re.match(r'^[\u0900-\u097F]+$', word):
        return "incorrect", "high", "non-Devanagari"

    # repetition
    if re.search(r'(.)\1{2,}', word):
        return "incorrect", "high", "repetition"

    # length issue
    if len(word) < 2 or len(word) > 20:
        return "incorrect", "medium", "length issue"

    # English-like
    if word in ["कंप्यूटर", "इंटरव्यू", "जॉब", "फिल्म"]:
        return "correct", "high", "English word"

    # LOW confidence (IMPORTANT)
    if len(word) <= 3:
        return "correct", "low", "very short word (uncertain)"

    return "correct", "medium", "looks valid"

In [11]:
results = []

for w in words_list:
    label, confidence, reason = classify_word(w)
    results.append((w, label, confidence, reason))

df_words = pd.DataFrame(results, columns=["word", "label", "confidence", "reason"])

In [12]:
low_conf = df_words[df_words["confidence"] == "low"]

sample_50 = low_conf.sample(50)
sample_50

,word,label,confidence,reason
113795,कचच,correct,low,very short word (uncertain)
119966,अउस,correct,low,very short word (uncertain)
81311,वपर,correct,low,very short word (uncertain)
26020,उला,correct,low,very short word (uncertain)
109496,एएह,correct,low,very short word (uncertain)
19042,जॉप,correct,low,very short word (uncertain)
49748,पाश,correct,low,very short word (uncertain)
21518,उना,correct,low,very short word (uncertain)
55893,नहे,correct,low,very short word (uncertain)
1384,राम,correct,low,very short word (uncertain)


In [13]:
def build_lattice(model_outputs, reference):
    aligned_sequences = align_all(model_outputs + [reference])

    lattice = []

    for position in range(len(aligned_sequences[0])):
        bin_words = set()

        for seq in aligned_sequences:
            word = seq[position]
            if word != "<blank>":
                bin_words.add(word)

        lattice.append(list(bin_words))

    return lattice

In [1]:
def lattice_wer(prediction, lattice):
    pred_words = prediction.split()

    errors = 0

    for i, word in enumerate(pred_words):
        if i >= len(lattice):
            errors += 1
            continue

        if word not in lattice[i]:
            errors += 1

    return errors / len(lattice)

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Unique Words Data.xlsx to Unique Words Data.xlsx


In [6]:
import pandas as pd

df = pd.read_excel(list(uploaded.keys())[0])
df.head()

,word
0,है
1,तो
2,में
3,जी
4,हैं


In [8]:
words_list = df.iloc[:,0].astype(str).tolist()

In [9]:
import re

def classify_word(word):
    if not re.match(r'^[\u0900-\u097F]+$', word):
        return "incorrect spelling"

    if re.search(r'(.)\1{2,}', word):
        return "incorrect spelling"

    if len(word) < 2 or len(word) > 20:
        return "incorrect spelling"

    return "correct spelling"

In [11]:
labels = []

for w in words_list:
    labels.append(classify_word(w))

df_final = pd.DataFrame({
    "word": words_list,
    "label": labels
})

df_final.head()

,word,label
0,है,correct spelling
1,तो,correct spelling
2,में,correct spelling
3,जी,correct spelling
4,हैं,correct spelling


In [12]:
df_final.to_csv("final_words.csv", index=False)

In [14]:
from google.colab import files
files.download("final_words.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>